# Event Producer - Windows-style security logs

## Purpose
Simulate a real-world source of security-relevant events and inject them into Kafka **asynchronously**. The producer is intentionally ignorant of any downstream consumer - this is the whole point of an event-driven SOC pipeline.

## Events emitted
* `process_start` - Windows process execution log (powershell / cmd / notepad)
* `user_login`    - authentication outcome (success / failure)

Each event carries `event_id`, `timestamp`, `user`, `host`, `source_ip`, and type-specific fields.

## Tracing model
We use **OpenTelemetry** with a **trace_id derived from the event_id** so the consumer can reconstruct the parent context without the producer needing to push trace headers through Kafka. This lets Jaeger stitch the producer span + consumer span into one end-to-end trace.

In [ ]:
!pip install -q kafka-python

In [ ]:
!pip install -q opentelemetry-sdk

In [ ]:
!pip install -q opentelemetry-exporter-otlp

In [ ]:
import json
import random
import time
import uuid
from datetime import datetime, timezone

from kafka import KafkaProducer

# OpenTelemetry imports
from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.trace import SpanKind, TraceFlags
from opentelemetry.trace import set_span_in_context
from opentelemetry.trace import SpanContext, NonRecordingSpan

# Configuration
KAFKA_BOOTSTRAP_SERVERS = "kafka:9092"
KAFKA_TOPIC = "raw-events"
OTLP_ENDPOINT = "jaeger:4317"
SERVICE_NAME = "windows-log-producer"
EVENT_INTERVAL_SEC = (1, 5)

# Tracing setup
resource = Resource.create({"service.name": SERVICE_NAME})
trace.set_tracer_provider(TracerProvider(resource=resource))
tracer = trace.get_tracer(__name__)

otlp_exporter = OTLPSpanExporter(endpoint=OTLP_ENDPOINT, insecure=True)
span_processor = BatchSpanProcessor(otlp_exporter)
trace.get_tracer_provider().add_span_processor(span_processor)

# Kafka producer
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)

# Event generation data
USERS = ["alice", "bob", "charlie", "admin"]
HOSTS = ["win-01", "win-02", "win-03"]
IPS = ["10.0.0.10", "10.0.0.11", "10.0.0.12"]

PROCESS_SCENARIOS = [
    {
        "process_name": "powershell.exe",
        "command_line": "powershell -EncodedCommand SQBFAFgA",
        "parent_process": "explorer.exe",
    },
    {
        "process_name": "cmd.exe",
        "command_line": "cmd.exe /c whoami",
        "parent_process": "explorer.exe",
    },
    {
        "process_name": "notepad.exe",
        "command_line": "notepad.exe",
        "parent_process": "explorer.exe",
    },
]

LOGIN_SCENARIOS = [
    {"logon_type": "success"},
    {"logon_type": "failure"},
]

def generate_event():
    event_id = str(uuid.uuid4())
    event_type = random.choice(["process_start", "user_login"])

    event = {
        "event_id": event_id,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "user": random.choice(USERS),
        "host": random.choice(HOSTS),
        "source_ip": random.choice(IPS),
        "event_type": event_type,
    }

    if event_type == "process_start":
        s = random.choice(PROCESS_SCENARIOS)
        event.update(
            {
                "process_name": s["process_name"],
                "command_line": s["command_line"],
                "parent_process": s["parent_process"],
                "logon_type": None,
            }
        )
    else:
        s = random.choice(LOGIN_SCENARIOS)
        event.update(
            {
                "process_name": None,
                "command_line": None,
                "parent_process": None,
                "logon_type": s["logon_type"],
            }
        )

    return event

# Main loop
print("Starting Windows log producer (OTLP)...")
print(f"Kafka topic: {KAFKA_TOPIC}")
print(f"OTLP endpoint: {OTLP_ENDPOINT}")

while True:
    event = generate_event()
    event_id = event["event_id"]
    trace_id = uuid.UUID(event_id).int

    parent_ctx = set_span_in_context(
        NonRecordingSpan(
            SpanContext(
                trace_id=trace_id,
                span_id=random.getrandbits(64),
                is_remote=False,
                trace_flags=TraceFlags(TraceFlags.SAMPLED),
                trace_state={},
            )
        )
    )

    with tracer.start_as_current_span(
        "produce_event",
        context=parent_ctx,
        kind=SpanKind.PRODUCER,
    ) as span:

        span.set_attribute("event.id", event_id)
        span.set_attribute("event.type", event["event_type"])
        span.set_attribute("host.name", event["host"])
        span.set_attribute("user.name", event["user"])

        with tracer.start_as_current_span("kafka_produce"):
            producer.send(KAFKA_TOPIC, event)
            producer.flush()

        print(f"Produced event {event_id} ({event['event_type']})")

    time.sleep(random.uniform(*EVENT_INTERVAL_SEC))